# Day 048 — Exercise 1: prepare_features + split_data

**What you'll build:** Two functions that start every ML pipeline:
- `prepare_features(df, target_col, numeric_only=True) -> (X, y)` — separate features from the target column
- `split_data(X, y, test_size=0.2, random_state=42) -> dict` — wrap `train_test_split` and return a labelled result dict

**Why it matters:** The first rule of ML: never let the model see test data during training. `split_data` enforces that boundary. `prepare_features` enforces the X/y interface that every sklearn estimator expects — a 2D feature matrix and a 1D target vector.

## Provided: Setup + Sample Data

In [ ]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')


def make_regression_data(n: int = 200, seed: int = 42) -> pd.DataFrame:
    """Synthetic housing dataset with one categorical column (neighborhood)."""
    rng = np.random.default_rng(seed)
    area         = rng.uniform(500, 3000, n).round(0)
    bedrooms     = rng.integers(1, 6, n)
    age          = rng.uniform(0, 50, n).round(1)
    neighborhood = rng.choice(['downtown', 'suburb', 'rural'], n)
    price = (
        area * 150
        + bedrooms * 10_000
        - age * 1_000
        + np.where(neighborhood == 'downtown', 50_000, 0)
        + np.where(neighborhood == 'suburb',   20_000, 0)
        + rng.standard_normal(n) * 10_000
    ).round(-2)
    return pd.DataFrame({
        'area':         area.astype(int),
        'bedrooms':     bedrooms,
        'age':          age,
        'neighborhood': neighborhood,
        'price':        price.astype(int),
    })

## Your Implementation

In [ ]:
def prepare_features(df: pd.DataFrame, target_col: str,
                     numeric_only: bool = True):
    """
    Separate features (X) from the target (y).

    Args:
        df:           input DataFrame
        target_col:   name of the column to predict
        numeric_only: if True, keep only numeric feature columns
    Returns:
        (X, y) — feature DataFrame and target Series
    """
    # TODO: X = df.drop(columns=[target_col])
    # TODO: if numeric_only:
    #     X = X.select_dtypes(include='number')
    # TODO: y = df[target_col]
    # TODO: return X, y
    pass


def split_data(X: pd.DataFrame, y: pd.Series,
               test_size: float = 0.2,
               random_state: int = 42) -> dict:
    """
    Split X and y into train and test sets.

    Returns a dict with keys:
        X_train, X_test, y_train, y_test, n_train, n_test, n_features
    """
    # TODO: X_train, X_test, y_train, y_test = train_test_split(
    #     X, y, test_size=test_size, random_state=random_state
    # )
    # TODO: return {
    #     'X_train': X_train, 'X_test': X_test,
    #     'y_train': y_train, 'y_test': y_test,
    #     'n_train': len(X_train), 'n_test': len(X_test),
    #     'n_features': X_train.shape[1],
    # }
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    df = make_regression_data(100)

    # Check 1: prepare_features defined, returns 2-tuple
    try:
        assert 'prepare_features' in globals()
        result = prepare_features(df, 'price')
        assert isinstance(result, tuple) and len(result) == 2, \
            f'expected 2-tuple, got {type(result).__name__}'
        passed += 1; print('\u2705 Check 1: prepare_features returns (X, y) tuple')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: X is DataFrame, y is Series
    try:
        X, y = prepare_features(df, 'price')
        assert isinstance(X, pd.DataFrame), \
            f'X must be DataFrame, got {type(X).__name__}'
        assert isinstance(y, pd.Series), \
            f'y must be Series, got {type(y).__name__}'
        passed += 1; print(f'\u2705 Check 2: X is DataFrame ({X.shape}), y is Series (len={len(y)})')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: target_col not in X.columns
    try:
        assert 'price' not in X.columns, \
            'target column should not appear in X'
        passed += 1; print(f'\u2705 Check 3: target col excluded from X (X cols: {X.columns.tolist()})')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: split_data returns dict with all required keys
    try:
        assert 'split_data' in globals()
        split = split_data(X, y)
        assert isinstance(split, dict), \
            f'split_data must return dict, got {type(split).__name__}'
        for k in ('X_train', 'X_test', 'y_train', 'y_test',
                  'n_train', 'n_test', 'n_features'):
            assert k in split, f'missing key: {k!r}'
        passed += 1; print(f'\u2705 Check 4: split_data returns dict with all 7 keys')
    except Exception as e:
        print(f'\u274c Check 4: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 5: n_train + n_test == len(X) — no rows lost
    try:
        total_rows = split['n_train'] + split['n_test']
        assert total_rows == len(X), \
            f'n_train + n_test should be {len(X)}, got {total_rows}'
        passed += 1; print(f'\u2705 Check 5: n_train={split["n_train"]} + n_test={split["n_test"]} = {len(X)} (no rows lost)')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def prepare_features(df: pd.DataFrame, target_col: str,
                     numeric_only: bool = True):
    """Return (X, y) separating features from target."""
    X = df.drop(columns=[target_col])
    if numeric_only:
        X = X.select_dtypes(include='number')
    y = df[target_col]
    return X, y


def split_data(X: pd.DataFrame, y: pd.Series,
               test_size: float = 0.2,
               random_state: int = 42) -> dict:
    """Wrap train_test_split, return a result dict."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    return {
        'X_train':    X_train,
        'X_test':     X_test,
        'y_train':    y_train,
        'y_test':     y_test,
        'n_train':    len(X_train),
        'n_test':     len(X_test),
        'n_features': X_train.shape[1],
    }
```

</details>